In [2]:
import numpy as np

def evaluate_and_rank_team_fitness(population, objective_function):
    """
    Evaluates and ranks a group/team of agents based on fitness metrics.
    Maps exactly to Step 4 of the optimization framework:
      - Leader   (Equivalent to Alpha α)  -> Best fitness score
      - Mentor   (Equivalent to Beta β)   -> Second-best fitness score
      - Advisor  (Equivalent to Delta δ)  -> Third-best fitness score
    """
    # Initialize tier metrics to infinity for a minimization task
    leader_score = float("inf")
    mentor_score = float("inf")
    advisor_score = float("inf")

    leader_pos = None
    mentor_pos = None
    advisor_pos = None

    # 1. Evaluate fitness f(x) for each individual in the group
    for i in range(len(population)):
        current_position = population[i]

        # Run the fitness evaluation formula
        fitness_score = objective_function(current_position)

        # 2. Sequential Leaderboard Ranking Loop
        if fitness_score < leader_score:
            # Shift previous ranks down
            advisor_score = mentor_score
            advisor_pos = mentor_pos.copy() if mentor_pos is not None else None

            mentor_score = leader_score
            mentor_pos = leader_pos.copy() if leader_pos is not None else None

            # Promote to Leader
            leader_score = fitness_score
            leader_pos = current_position.copy()

        elif fitness_score < mentor_score:
            # Shift to Advisor
            advisor_score = mentor_score
            advisor_pos = mentor_pos.copy() if mentor_pos is not None else None

            # Promote to Mentor
            mentor_score = fitness_score
            mentor_pos = current_position.copy()

        elif fitness_score < advisor_score:
            # Promote to Advisor
            advisor_score = fitness_score
            advisor_pos = current_position.copy()

    # Returns tuples: (position, score)
    return (leader_pos, leader_score), (mentor_pos, mentor_score), (advisor_pos, advisor_score)


# --- TARGET FITNESS FUNCTION CRITERIA ---
def mathematical_fitness_evaluation(x):
    """Calculates a standard multi-variable continuous Sphere function."""
    return np.sum(x ** 2)


# --- INTERACTIVE EXECUTION HARNESS ---
if __name__ == "__main__":
    print("==========================================================")
    print("      TEAM OPTIMIZATION: PARAMETERS & INITIALIZATION      ")
    print("==========================================================")

    # Collect Interactive User Inputs
    try:
        group_size  = int(input("Enter number of individuals (Population size N, e.g., 6): ") or 6)
        dimensions  = int(input("Enter space dimensions (Variables count, e.g., 2): ") or 2)
        lower_bound = float(input("Enter Lower Bound fence limit (e.g., -10): ") or -10)
        upper_bound = float(input("Enter Upper Bound fence limit (e.g., 10): ") or 10)
    except ValueError:
        print("\n[Input Notice] Invalid numeric format. Loading fallback defaults...")
        group_size, dimensions, lower_bound, upper_bound = 6, 2, -10, 10

    # Initialize random starting coordinate positions bounded inside the playground space
    np.random.seed(42)
    initial_population = np.random.uniform(lower_bound, upper_bound, (group_size, dimensions))

    print("\nInitial Group Coordinate Positions Grid:")
    for idx, person in enumerate(initial_population):
        print(f"  Individual {idx}: {np.round(person, 4)}")

    print("\n[Processing] Running Fitness Evaluations...")

    # Execute step 4 fitness evaluation matrix routing
    leader_tuple, mentor_tuple, advisor_tuple = evaluate_and_rank_team_fitness(initial_population, mathematical_fitness_evaluation)

    # Unpack the positions and scores properly to avoid the NumPy round error
    leader_pos, leader_score = leader_tuple
    mentor_pos, mentor_score = mentor_tuple
    advisor_pos, advisor_score = advisor_tuple

    print("\n==================== RANKED HIERARCHY MATRIX ====================")
    print(f"🥇 Leader  (Best Guess)    | Pos: {np.round(leader_pos, 4)} | Fitness: {leader_score:.4f}")
    print(f"🥈 Mentor  (2nd Best)     | Pos: {np.round(mentor_pos, 4)} | Fitness: {mentor_score:.4f}")
    print(f"🥉 Advisor (3rd Best)     | Pos: {np.round(advisor_pos, 4)} | Fitness: {advisor_score:.4f}")
    print("=================================================================")


      TEAM OPTIMIZATION: PARAMETERS & INITIALIZATION      
Enter number of individuals (Population size N, e.g., 6): 6
Enter space dimensions (Variables count, e.g., 2): 2
Enter Lower Bound fence limit (e.g., -10): -10
Enter Upper Bound fence limit (e.g., 10): 10

Initial Group Coordinate Positions Grid:
  Individual 0: [-2.5092  9.0143]
  Individual 1: [4.6399 1.9732]
  Individual 2: [-6.8796 -6.8801]
  Individual 3: [-8.8383  7.3235]
  Individual 4: [2.0223 4.1615]
  Individual 5: [-9.5883  9.3982]

[Processing] Running Fitness Evaluations...

==================== RANKED HIERARCHY MATRIX ====================
🥇 Leader  (Best Guess)    | Pos: [2.0223 4.1615] | Fitness: 21.4074
🥈 Mentor  (2nd Best)     | Pos: [4.6399 1.9732] | Fitness: 25.4219
🥉 Advisor (3rd Best)     | Pos: [-2.5092  9.0143] | Fitness: 87.5534
